# Curate two multispectral datasets

In this tutorial, you will audit two object-detection datasets against a shared label space,
conform and merge them, and then clean, profile, and assess the combined dataset from a
single configuration file.

**Who this is for**: Engineers combining datasets collected under different labeling
conventions who need a unified, defensible dataset.

**Where this fits**: Run this curation step before model training. It verifies that source
label spaces resolve against a shared ontology, assesses what is lost during conforming,
and checks that the merged dataset is fit for downstream evaluation.

## What you'll do

In this tutorial, you will:
1. **Inspect the datasets**: Check two RGB/infrared datasets with differing class vocabularies.
2. **Review configuration**: Inspect the pipeline declared in `curation_demo.yaml`.
3. **Run the pipeline**: Execute eight evaluation tasks in a single call.
4. **Audit label spaces**: Reconcile class vocabularies and check conformance costs.
5. **Clean the merged dataset**: Detect outliers and duplicates across named band groups.
6. **Profile the data**: Compute balance and diversity from multispectral image statistics.
7. **Assess coverage**: Measure embedding coverage across visible and thermal bands.
8. **Split and verify**: Partition into stratified splits and verify cross-dataset parity.

## What you'll learn

You will learn:
- How `ontologies:` entries detect and report label mismatches
- How to extend external ontology artifacts directly from configuration
- Why cleanly aligned datasets can still produce lossy or partial merges
- How `channel_groups:` and stats policies measure multispectral bands independently
- How embedding coverage varies across different sensor bands on the same images

## What you'll need

- `dataeval-flow` (includes `dataeval`, `datamaite`, and `pydantic`)
- `dataeval-flow[ontology]` (`rdflib` for JSON-LD ontology support)
- `maite-datasets` (provides M3FD and DroneVehicle)
- Both datasets downloaded under `./data`. Because Kaggle credentials are required, download
  them beforehand using `M3FD("data", download=True)` and `DroneVehicle("data", download=True)`.
  The tutorial config sets `download: false`.

## Step 1: Check the data

[M3FD](https://github.com/JinyuanLiu-CV/TarDAL) and
[DroneVehicle](https://github.com/VisDrone/DroneVehicle) pair visible-light frames with
thermal frames in a four-channel format (three RGB channels and one infrared). While their
domains overlap, their class vocabularies differ.

In [1]:
from maite_datasets.object_detection import M3FD, DroneVehicle

print("M3FD:        ", list(M3FD.index2label.values()))
print("DroneVehicle:", list(DroneVehicle.index2label.values()))

M3FD:         ['people', 'car', 'bus', 'motorcycle', 'lamp', 'truck']
DroneVehicle: ['car', 'truck', 'bus', 'van', 'freight car']


Two mismatches require different remediation strategies:

- **`freight car`**: A land vehicle missing from the target ontology. Add it to the label
  space in configuration.
- **`lamp`**: A street fixture outside the vehicle and person domain. Leave it unmapped so
  conformance drops it.

All other classes (`car`, `bus`, `truck`, `van`, `motorcycle`, `people`) resolve directly.

In [2]:
from pathlib import Path

m3fd = M3FD("data", image_set="train", download=False)
drone = DroneVehicle("data", image_set="test", download=False)

image, target, _ = m3fd[0]
print(f"M3FD:         {len(m3fd)} images, shape {image.shape} (3 visible + 1 infrared)")
print(f"DroneVehicle: {len(drone)} images")

M3FD:         4200 images, shape (4, 768, 1024) (3 visible + 1 infrared)
DroneVehicle: 8980 images


## Step 2: Read the configuration

The pipeline is defined in a single YAML file using modular, reusable sections:

| Section | Purpose in this pipeline |
| --- | --- |
| `datasets` | Dataset paths and channel definitions (channel 3 as infrared) |
| `ontologies` | Shared label space extended with the missing `Freight Car` concept |
| `views` | Subsampling, band selections, and conforming `Relabel` operations |
| `stats` | Policies specifying which statistics to measure per band group |
| `metadata` | Shared factor binning to keep bias metrics comparable |
| `sources` | Source definitions and the conformed merge combining both datasets |
| `extractors` | Feature extractor configuration for image embeddings |
| `workflows` | Parameter presets for each evaluation workflow |
| `tasks` | Task definitions binding workflows to sources and extractors |

In [3]:
config_path = Path("curation_demo.yaml")
print(config_path.read_text())

# Dataset curation: audit two multispectral datasets, conform them, merge, and evaluate
#
# This is the single source of truth for the `curation_demo` tutorial. The notebook
# loads this exact file, and the container runs this exact file. Nothing in the
# pipeline is defined in Python.
#
# All paths are relative to the data root:
#   * notebook:  docs/source/notebooks/
#   * container: the directory bind-mounted at /dataeval
#
# Run it locally:
#   dataeval-flow --config curation_demo.yaml --data . --output ./output --cache ./cache -v
#
# M3FD and DroneVehicle are credential-gated, so `download: false` below expects them
# already under `data/`. See the tutorial for how to get them.
#
# yaml-language-server: $schema=../../../config/params.schema.json

# Pin every stochastic component (sampling, clustering, splitting) so re-runs match.
seed: 42

# -----------------------------------------------------------------------------
# DATASETS: where the data lives, and the facts about how it wa

### Key configuration details

**Extend ontologies without editing source files**: Use `concepts:` to add `Freight Car`
to the loaded ontology. Because declared concepts replace existing definitions with the
same id, you must restate `parents` to avoid detaching the concept into an unlinked root.

**Match target vocabularies before merging**: `merge_datasets` requires identical
`index2label` mappings across operands. Conforming both datasets to the same target
vocabulary (including abstract parent concepts) ensures class indices align across sources.

**Declare measurement scopes explicitly**: Views omitted from `measure` are not measured.
For example, `dimension` is measured only on the whole image (`~`), because geometric dimensions
do not vary across channel subsets.

In [4]:
from dataeval_flow import load_config

config = load_config(config_path)

print(f"Datasets:   {[d.name for d in config.datasets]}")
print(f"Ontologies: {[o.name for o in config.ontologies]}")
print(f"Sources:    {[s.name for s in config.sources]}")
print(f"Workflows:  {[(w.name, w.type) for w in config.workflows]}")
print(f"Tasks:      {[t.name for t in config.tasks]}")

Datasets:   ['m3fd', 'drone_vehicle']
Ontologies: ['vehicles']
Sources:    ['m3fd_raw', 'drone_raw', 'm3fd_conf', 'drone_conf', 'merged', 'merged_rgb', 'merged_ir']
Workflows:  [('audit', 'data-coverage'), ('clean', 'data-cleaning'), ('profile', 'data-analysis'), ('gaps', 'data-coverage'), ('verify', 'data-analysis'), ('split', 'data-splitting')]
Tasks:      ['audit_m3fd', 'audit_drone', 'clean_merged', 'profile_merged', 'gaps_rgb', 'gaps_ir', 'split_merged', 'verify_datasets']


## Step 3: Run the pipeline

`run_tasks()` executes all enabled tasks in sequence. Tasks reading the merged dataset
share cache entries, computing underlying statistics once and reusing them across workflows.

In [5]:
from dataeval_flow import run_tasks

results = run_tasks(config, data_dir=Path("."), cache_dir=Path("./cache"))

/home/aweng/2033/dataeval-flow/src/dataeval_flow/workflows/cleaning/workflow.py:268: UserWarning: Declared cuts left bins unused: unit_ir_darkness (4 of 5 bins hold rows). The encoding still applies and the codes are unchanged; this is the data no longer matching the policy, not an error. Call reencode() to refit — which moves codes — or leave it and read the empty groups as the finding they are.
  td = metadata.rows_at(metadata.label_level).select("item_index", "target_index", "class_label")


/home/aweng/2033/dataeval-flow/src/dataeval_flow/binning.py:164: UserWarning: Declared cuts left bins unused: unit_ir_darkness (4 of 5 bins hold rows). The encoding still applies and the codes are unchanged; this is the data no longer matching the policy, not an error. Call reencode() to refit — which moves codes — or leave it and read the empty groups as the finding they are.
  export(path)


/home/aweng/2033/dataeval-flow/src/dataeval_flow/cache.py:282: UserWarning: Declared cuts left bins unused: instance_ir_brightness (0 of 5 bins hold rows), instance_ir_contrast (0 of 5 bins hold rows), instance_ir_darkness (0 of 5 bins hold rows), instance_ir_sharpness (0 of 5 bins hold rows), unit_ir_brightness (0 of 5 bins hold rows), unit_ir_contrast (0 of 5 bins hold rows), unit_ir_darkness (0 of 5 bins hold rows), unit_ir_sharpness (0 of 5 bins hold rows). The encoding still applies and the codes are unchanged; this is the data no longer matching the policy, not an error. Call reencode() to refit — which moves codes — or leave it and read the empty groups as the finding they are.
  export(path)


/home/aweng/2033/dataeval-flow/src/dataeval_flow/cache.py:282: UserWarning: Declared cuts left bins unused: instance_ir_brightness (0 of 5 bins hold rows), instance_ir_contrast (0 of 5 bins hold rows), instance_ir_darkness (0 of 5 bins hold rows), instance_ir_sharpness (0 of 5 bins hold rows), instance_rgb_brightness (0 of 5 bins hold rows), instance_rgb_contrast (0 of 5 bins hold rows), instance_rgb_darkness (0 of 5 bins hold rows), instance_rgb_sharpness (0 of 5 bins hold rows), unit_darkness (4 of 5 bins hold rows), unit_ir_brightness (0 of 5 bins hold rows), unit_ir_contrast (0 of 5 bins hold rows), unit_ir_darkness (0 of 5 bins hold rows), unit_ir_sharpness (0 of 5 bins hold rows), unit_rgb_brightness (0 of 5 bins hold rows), unit_rgb_contrast (0 of 5 bins hold rows), unit_rgb_darkness (0 of 5 bins hold rows), unit_rgb_sharpness (0 of 5 bins hold rows). The encoding still applies and the codes are unchanged; this is the data no longer matching the policy, not an error. Call reenco

/home/aweng/2033/dataeval-flow/src/dataeval_flow/metadata.py:243: UserWarning: `infrared_filename` and `rgb_filename` were dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  per_target=metadata.multi_target,


/home/aweng/2033/dataeval-flow/src/dataeval_flow/cache.py:282: UserWarning: Declared cuts left bins unused: unit_contrast (1 of 5 bins hold rows), unit_darkness (1 of 5 bins hold rows), unit_ir_brightness (2 of 5 bins hold rows), unit_ir_contrast (2 of 5 bins hold rows), unit_ir_darkness (1 of 5 bins hold rows), unit_ir_sharpness (1 of 5 bins hold rows), unit_rgb_contrast (3 of 5 bins hold rows), unit_rgb_darkness (1 of 5 bins hold rows), unit_sharpness (4 of 5 bins hold rows). The encoding still applies and the codes are unchanged; this is the data no longer matching the policy, not an error. Call reencode() to refit — which moves codes — or leave it and read the empty groups as the finding they are.
  export(path)


In [6]:
for task, result in zip(config.tasks, results, strict=True):
    status = "OK " if result.success else "FAIL"
    warnings = sum(1 for f in result.data.report.findings if f.severity == "warning")
    elapsed = result.metadata.execution_time_s or 0.0
    print(f"[{status}] {task.name:<16} {result.name:<16} {elapsed:>6.1f}s  {warnings} warning(s)")
    for error in result.errors:
        print(f"         {error}")

[OK ] audit_m3fd       data-coverage       0.1s  4 warning(s)
[OK ] audit_drone      data-coverage       0.1s  2 warning(s)
[OK ] clean_merged     data-cleaning       0.4s  2 warning(s)
[OK ] profile_merged   data-analysis       0.9s  3 warning(s)
[OK ] gaps_rgb         data-coverage      92.9s  4 warning(s)
[OK ] gaps_ir          data-coverage      59.3s  4 warning(s)
[OK ] split_merged     data-splitting      8.7s  2 warning(s)
[OK ] verify_datasets  data-analysis     148.9s  6 warning(s)


In [7]:
assert all(r.success for r in results), [r.errors for r in results if not r.success]

In [8]:
by_name = dict(zip([t.name for t in config.tasks], results, strict=True))

## Step 4: Audit the label spaces

The audit tasks evaluate each raw dataset before conforming to determine whether source
class names resolve against the reference ontology.

In [9]:
for name in ("audit_m3fd", "audit_drone"):
    alignment = by_name[name].data.raw.ontology.alignment
    print(f"{name}: mergeability = {alignment.mergeability}")
    for c in alignment.correspondences:
        print(f"    {c.source:<14} {c.relation:<12} -> {c.target_label}")
    if alignment.unaligned_source:
        print(f"    unaligned    : {list(alignment.unaligned_source)}")
    print()

audit_m3fd: mergeability = partial
    people         equivalent   -> Person
    car            equivalent   -> Car
    bus            equivalent   -> Bus
    motorcycle     equivalent   -> Motorcycle
    truck          equivalent   -> Truck
    unaligned    : ['lamp']

audit_drone: mergeability = lossless
    truck          equivalent   -> Truck
    car            equivalent   -> Car
    freight car    equivalent   -> Freight Car
    bus            equivalent   -> Bus
    van            equivalent   -> Van



### Interpreting the audit verdicts

- **DroneVehicle (`lossless`)**: Every class maps one-to-one to the ontology. This occurs
  because configuration added `Freight Car`; against the unmodified ontology artifact,
  `freight car` would be unmatched.
- **M3FD (`partial`)**: The `lamp` class cannot map to the target ontology. The `partial`
  verdict confirms that merging will drop `lamp` annotations, ensuring this exclusion is
  intentional.

Task findings highlight these verdicts with standard severity indicators.

In [10]:
for name in ("audit_m3fd", "audit_drone"):
    findings = by_name[name].data.report.findings
    relevant = [f for f in findings if "Label" in f.title]
    print(name)
    for f in relevant:
        print(f"    [{f.severity:<7}] {f.title}")
    print()

audit_m3fd
    [warning] Label Distribution
    [warning] Label Space Coverage
    [warning] Label Conformance
    [warning] Label Alignment

audit_drone
    [warning] Label Distribution
    [warning] Label Space Coverage
    [ok     ] Label Conformance
    [ok     ] Label Alignment



## Step 5: Clean the merged dataset

`clean_merged` evaluates the concatenated dataset after each source is conformed via
`Relabel`, applying the `multispectral` stats policy to detect outliers and duplicates.

In [11]:
clean = by_name["clean_merged"]
print(clean.report(detailed=False))


  DATA CLEANING COMPLETE. DATASET: 200 ITEMS. MODE: ADVISORY.
  Timestamp:    2026-09-07T21:30:34.104182+00:00
  Duration:     0.36s
  Source:       merged (merge: m3fd[m3fd_conformed] + drone_vehicle[drone_conformed])
  Model:        bovw (bovw)
--------------------------------------------------------------------------------

  SUMMARY
  -------
  Image Outliers ...................................... 16 images (8.0%)  [!!]
  Classwise Outliers .............................. no outliers detected  [ok]
  Duplicates ............................. 0 exact (0.0%), 2 near (1.0%)  [..]
  Label Distribution ........... 8 classes, 200 items, imbalance 155.8:1  [!!]

  Health: 2 warning(s) [!!] — review flagged findings

--------------------------------------------------------------------------------
  METADATA FACTORS
--------------------------------------------------------------------------------
  Encoding:        741d5236f2b799c3
  Auto-bin method: uniform_width
  Policy:          3 of 27 f

### Multispectral band groups

Without `channel_groups:`, statistics collapse across all four channels, blending visible
light and thermal measurements. Defining channel groups isolates each modality: `rgb_*`
columns describe visible channels, `ir_*` describe infrared channels, and unprefixed columns
describe the full four-channel image. Setting `background: true` measures unannotated scene
pixels outside bounding boxes.

The factors below show what `factors_from: [~, rgb, ir]` provides to the subsequent profiling task.

In [12]:
binning = by_name["profile_merged"].metadata.metadata_binning or {}

# Binning records are keyed by split to allow independent bin edges across partitions.
# In this task, the merged dataset is the sole split.
if "per_split" in binning:
    binning = next(iter(binning["per_split"].values()))
injected = sorted(binning.get("injected_factors", []))

# `add_factors` prefixes statistics by row level (`unit_` for images, `instance_` for
# detections). Strip the prefix to identify the source view.
groups: dict[str, list[str]] = {"whole image": [], "rgb": [], "ir": [], "background": []}
for factor in injected:
    stat = factor.split("_", 1)[1] if factor.startswith(("unit_", "instance_")) else factor
    if stat.startswith("background"):
        groups["background"].append(factor)
    elif stat.startswith("rgb_"):
        groups["rgb"].append(factor)
    elif stat.startswith("ir_"):
        groups["ir"].append(factor)
    else:
        groups["whole image"].append(factor)

print(f"{len(injected)} injected factors\n")
for view, names in groups.items():
    shown = ", ".join(names[:4])
    print(f"{view:<12} {len(names):>3}  {shown}{' ...' if len(names) > 4 else ''}")

24 injected factors

whole image    8  instance_brightness, instance_contrast, instance_darkness, instance_sharpness ...
rgb            8  instance_rgb_brightness, instance_rgb_contrast, instance_rgb_darkness, instance_rgb_sharpness ...
ir             8  instance_ir_brightness, instance_ir_contrast, instance_ir_darkness, instance_ir_sharpness ...
background     0  


The background row contains zero injected factors because `factors_from: [~, rgb, ir]`
omits background views. While `background: true` calculates background metrics for outlier
detection, factors are only injected into metadata when explicitly requested in `factors_from`.

## Step 6: Profile the merged dataset

`profile_merged` injects computed image statistics as metadata factors to assess balance and
diversity. For datasets without domain metadata, `intrinsic_factors` allows you to profile
representations using image and sensor statistics alone.

In [13]:
print(by_name["profile_merged"].report(detailed=False))


  DATASET ANALYSIS COMPLETE. 1 SPLIT(S), 200 TOTAL ITEMS.
  Timestamp:    2026-09-07T21:30:40.266492+00:00
  Duration:     0.90s
  Source:       merged (merge: m3fd[m3fd_conformed] + drone_vehicle[drone_conformed])
--------------------------------------------------------------------------------

  SUMMARY
  -------
  Image Quality ................................... 16 outliers (16/200)  [!!]
  Redundancy ................................ 0 exact, 6 near duplicates  [..]
  Label Balance ........................... 7 classes, imbalance 155.8:1  [!!]
  Bias ................................ 27 factors checked, issues found  [!!]

  Health: 3 warning(s) [!!] — review flagged findings

--------------------------------------------------------------------------------
  METADATA FACTORS
--------------------------------------------------------------------------------
  [merged]
  Encoding:        741d5236f2b799c3
  Auto-bin method: uniform_width
  Policy:          3 of 27 factors still derived 

## Step 7: Assess coverage through each band group

Tasks `gaps_rgb` and `gaps_ir` evaluate the same merged dataset using different channel views:
RGB selects channels 0–2, while IR replicates channel 3 across three channels. This measures
how embedding coverage varies between visual and thermal features on identical annotations.

In [14]:
for name in ("gaps_rgb", "gaps_ir"):
    coverage = by_name[name].data.raw.coverage
    print(f"{name}:")
    if coverage is None:
        print(f"    skipped: {by_name[name].data.raw.coverage_skipped_reason}")
        continue
    print(f"    crops measured     : {coverage.observation_count} ({coverage.observation_unit})")
    print(f"    dropped detections : {coverage.dropped_detections}")
    print(f"    uncovered          : {coverage.uncovered_count} ({coverage.uncovered_rate:.2%})")
    print(f"    coverage radius    : {coverage.coverage_radius:.4f}")
    print()

gaps_rgb:
    crops measured     : 2661 (detection crop)
    dropped detections : 59
    uncovered          : 26 (0.98%)
    coverage radius    : 2.3346

gaps_ir:
    crops measured     : 2661 (detection crop)
    dropped detections : 59
    uncovered          : 26 (0.98%)
    coverage radius    : 2.5904



The `crop_min_size` parameter filters out bounding boxes too small to extract meaningful
embeddings. Tracking these in `dropped_detections` clarifies what proportion of annotations
the coverage assessment excludes.

While aggregate coverage rates are similar between the two modalities, per-class feature
distributions differ substantially between visible and infrared bands.

In [15]:
rgb_rows = {r.class_name: r for r in by_name["gaps_rgb"].data.raw.coverage.per_class}
ir_rows = {r.class_name: r for r in by_name["gaps_ir"].data.raw.coverage.per_class}

print(f"{'class':<14} {'n':>5}   {'rgb disp':>9} {'ir disp':>9}   {'rgb iso':>8} {'ir iso':>8}")
for name in sorted(rgb_rows, key=lambda k: -rgb_rows[k].count):
    rgb, ir = rgb_rows[name], ir_rows.get(name)
    if ir is None or rgb.dispersion is None or ir.dispersion is None:
        continue
    print(
        f"{name:<14} {rgb.count:>5}   {rgb.dispersion:>9.3f} {ir.dispersion:>9.3f}   "
        f"{rgb.isotropy:>8.3f} {ir.isotropy:>8.3f}"
    )

class              n    rgb disp   ir disp    rgb iso   ir iso
Car             1996       1.023     1.128      0.534    0.563
Person           309       1.201     1.160      0.487    0.467
Truck            183       0.681     0.963      0.534    0.506
Bus               58       1.009     0.843      0.566    0.507
Freight Car       56       0.716     0.881      0.732    0.635
Van               47       0.991     1.037      0.778    0.766


Dispersion measures how widely class samples spread in feature space, while isotropy
measures directional uniformity. A class that appears compact and uniform in visible light
can be diffuse or directional in thermal imagery. Evaluating both modalities confirms
whether representations remain balanced across all operational sensor bands.

### Representation against the ontology

While coverage measures sample spread in feature space, representation analysis checks
sample counts against target ontology thresholds, identifying underrepresented or missing classes.

In [16]:
representation = by_name["gaps_rgb"].data.raw.ontology.representation
print(f"leaf coverage : {representation.leaf_coverage:.2f}")
print(f"total deficit : {representation.total_deficit}")
print()
for row in representation.worklist[:8]:
    print(f"    [{row.action:<7}] {row.label:<14} have {row.count:>4}, want {row.target:>4}")

leaf coverage : 1.00
total deficit : 1639

    [augment] Motorcycle     have   13, want  389
    [augment] Van            have   47, want  389
    [augment] Bus            have   58, want  389
    [augment] Freight Car    have   58, want  389
    [augment] Truck          have  184, want  389
    [augment] Person         have  335, want  389


## Step 8: Split and verify

`split_merged` partitions the merged dataset into stratified train, validation, and test splits.

In [17]:
print(by_name["split_merged"].report(detailed=False))


  DATASET SPLITTING: 200 ITEMS → 1 FOLD(S)
  Timestamp:    2026-09-07T21:33:29.365364+00:00
  Duration:     8.73s
  Source:       merged (merge: m3fd[m3fd_conformed] + drone_vehicle[drone_conformed])
--------------------------------------------------------------------------------

  SUMMARY
  -------
  Class distribution (full dataset) ...................................   [!!]
  Fold 0 split sizes ..................................................   [..]
  Class distribution across splits ....................................   [..]
  Stratification quality ................ WARNING - max deviation 19.2pp  [!!]
  Pre-split balance (mutual information) ..............................   [..]
  Pre-split diversity .................................................   [..]

  Health: 2 warning(s) [!!] — review flagged findings

  CONFIGURATION
  sources:
    - name: merged
      merge:
        - dataset: m3fd
          dataset_config:
            format: demo
            name: m3fd
          

`verify_datasets` tests label parity and feature-space divergence between the two conformed datasets.

:::{note}
In this step, verification runs across the two pre-merge source datasets to confirm they are
comparable. The same statistical checks apply when verifying split partitions.
:::

Label parity is evaluated only across classes present in both datasets. Classes unique to one
dataset cannot be evaluated with chi-squared tests and are reported separately under `label_overlap`.

In [18]:
for pair, cross in by_name["verify_datasets"].data.raw.cross_split.items():
    print(f"{pair}")
    overlap = cross.label_health.label_overlap if cross.label_health else {}
    print(f"    shared classes : {overlap.get('shared_classes')}")
    for key in overlap:
        if key.endswith("_only") and overlap[key]:
            print(f"    {key:<14} : {overlap[key]}")

    parity = cross.label_health.label_parity if cross.label_health else None
    if parity is not None:
        print(
            f"    label parity   : p = {parity['p_value']:.4g}, "
            f"significant = {parity['significant']} "
            f"({parity['classes_compared']} compared, {parity['classes_excluded']} excluded)"
        )
    if cross.distribution_shift is not None:
        print(f"    divergence     : {cross.distribution_shift}")
    print()

m3fd_conf_vs_drone_conf
    shared classes : ['Bus', 'Car', 'Truck']
    m3fd_conf_only : ['Motorcycle', 'Person']
    drone_conf_only : ['Van', 'Freight Car']
    label parity   : p = 3.073e-56, significant = True (3 compared, 4 excluded)
    divergence     : divergence=1.0 divergence_method='fnn'



## Run the pipeline from the command line

Because the entire workflow is declared in `curation_demo.yaml`, you can execute the same
pipeline directly from the command line:

```bash
dataeval-flow --config curation_demo.yaml --data . --output ./output --cache ./cache -v
```

The identical configuration runs inside Docker containers. See {doc}`../how_to/containerized_workflows`.

## Summary

In this tutorial, you:
- Audited two multispectral datasets against a reference ontology, extending missing concepts
  and dropping out-of-domain classes
- Conformed both datasets to a unified vocabulary and merged them into a single source
- Evaluated visible and infrared channels independently using `channel_groups` and stats policies
- Cleaned outliers, profiled balance and diversity, and measured coverage across sensor modalities
- Partitioned the merged data into stratified splits and verified cross-dataset parity

Declaring ontologies, stats policies, and metadata encodings once in configuration ensures
consistency across all downstream evaluation tasks.

### Where to go next

- {doc}`../how_to/declare_an_ontology`: Declare hierarchies, load RDF artifacts, and set expected distributions
- {doc}`../how_to/measure_band_groups`: Configure custom channel views and statistic policies
- {doc}`../how_to/export_a_dataset`: Export merged datasets to COCO, YOLO, or Hugging Face formats
- {doc}`end_to_end`: Run end-to-end evaluation on standard single-modality datasets